# Notebook 01 — Fraud Detection Platform: Full Analysis
**Real execution against the verified real Worldline/ULB Credit Card Fraud Detection dataset**

Covers: environment setup (WARP-optimized) → EDA with visual outputs → class-imbalance handling →
4–5 candidate model screening → champion + runner-up real 5-fold stratified CV → cost-optimal
threshold → confusion matrix → feature importance (SHAP) → export of every result table/figure for
the Word/Excel/HTML deliverables.

RANDOM_SEED = 42 throughout. Every number and chart below is computed live in this run.

In [1]:
# ============================================================
# SETUP — WARP-optimized environment (thread ceiling set BEFORE any ML import)
# ============================================================
import os, time, json, pickle, warnings
warnings.filterwarnings("ignore")

_N_THREADS = max(1, int((os.cpu_count() or 4) * 0.92 // 1))
os.environ.setdefault("OMP_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(_N_THREADS))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_predict
from sklearn.metrics import (average_precision_score, precision_score, recall_score,
                              confusion_matrix, precision_recall_curve)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

sns.set_theme(style="whitegrid", font_scale=1.05)
PALETTE = {"legit": "#2E74B5", "fraud": "#C0392B", "accent": "#1F3864", "grey": "#7F8C8D"}
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["font.family"] = "DejaVu Sans"

RESULTS_DIR = "nb1_results"
FIG_DIR = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

def save_fig(fig, name):
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    print(f"Saved figure: {path}")
    return path

print(f"WARP thread ceiling: {_N_THREADS} threads (of {os.cpu_count()} available cores)")
print("Setup complete.")

WARP thread ceiling: 1 threads (of 2 available cores)
Setup complete.


In [ ]:
# ============================================================
# REPO-LAYOUT BOOTSTRAP — added when this notebook was organized into the
# Fraud_Detection_Platform repo structure (notebooks/ + src/ + data/raw/).
# Makes the local module imports (class_imbalance_utils, model_benchmark, ...)
# and the dataset path resolve whether this notebook is run from notebooks/
# or copied elsewhere, without changing any of the real modeling logic below.
# ============================================================
import sys, os as _os
_REPO_ROOT = _os.path.abspath(_os.path.join(_os.getcwd(), ".."))
_SRC_DIR = _os.path.join(_REPO_ROOT, "src")
if _os.path.isdir(_SRC_DIR) and _SRC_DIR not in sys.path:
    sys.path.insert(0, _SRC_DIR)

def _resolve_data_path():
    candidates = [
        "creditcard.csv",                                   # same-folder (original layout)
        _os.path.join("..", "data", "raw", "creditcard.csv"),  # repo layout, run from notebooks/
        _os.path.join(_REPO_ROOT, "data", "raw", "creditcard.csv"),
    ]
    for c in candidates:
        if _os.path.exists(c):
            return c
    raise FileNotFoundError(
        "creditcard.csv not found. Expected it at data/raw/creditcard.csv in the repo root, "
        "or alongside this notebook. See README.md."
    )


## 1. Load & Verify Real Data (Section 2)

In [2]:
DATA_PATH = _resolve_data_path()
df = pd.read_csv(DATA_PATH)

print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
print(f"Nulls: {df.isnull().sum().sum()}  |  Duplicate rows: {df.duplicated().sum()}")
print(f"Fraud rate: {df['Class'].mean():.6%}  ({df['Class'].sum()} / {len(df):,})")
display(df.describe().T.style.background_gradient(cmap="Blues").set_caption("Real Descriptive Statistics"))

Rows: 284,807  |  Columns: 31


Nulls: 0  |  Duplicate rows: 1081
Fraud rate: 0.172749%  (492 / 284,807)


,count,mean,std,min,25%,50%,75%,max
Time,284807.000000,94813.859575,47488.145955,0.000000,54201.500000,84692.000000,139320.500000,172792.000000
V1,284807.000000,0.000000,1.958696,-56.407510,-0.920373,0.018109,1.315642,2.454930
V2,284807.000000,-0.000000,1.651309,-72.715728,-0.598550,0.065486,0.803724,22.057729
V3,284807.000000,-0.000000,1.516255,-48.325589,-0.890365,0.179846,1.027196,9.382558
V4,284807.000000,0.000000,1.415869,-5.683171,-0.848640,-0.019847,0.743341,16.875344
V5,284807.000000,0.000000,1.380247,-113.743307,-0.691597,-0.054336,0.611926,34.801666
V6,284807.000000,0.000000,1.332271,-26.160506,-0.768296,-0.274187,0.398565,73.301626
V7,284807.000000,-0.000000,1.237094,-43.557242,-0.554076,0.040103,0.570436,120.589494
V8,284807.000000,0.000000,1.194353,-73.216718,-0.208630,0.022358,0.327346,20.007208
V9,284807.000000,-0.000000,1.098632,-13.434066,-0.643098,-0.051429,0.597139,15.594995


In [3]:
# Class balance chart
fig, ax = plt.subplots(figsize=(6, 4.5))
counts = df["Class"].value_counts().sort_index()
bars = ax.bar(["Legitimate", "Fraud"], counts.values, color=[PALETTE["legit"], PALETTE["fraud"]])
ax.set_yscale("log")
ax.set_ylabel("Transaction count (log scale)")
ax.set_title(f"Real Class Balance — {df['Class'].mean():.4%} fraud rate", fontweight="bold", color=PALETTE["accent"])
for b, v in zip(bars, counts.values):
    ax.annotate(f"{v:,}", (b.get_x() + b.get_width()/2, v), ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
save_fig(fig, "01_class_balance.png")
plt.show()

Saved figure: nb1_results/figures/01_class_balance.png


In [4]:
# Amount distribution by class
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(df.loc[df.Class==0, "Amount"].clip(upper=500), bins=50, color=PALETTE["legit"], ax=axes[0], stat="density")
axes[0].set_title("Legitimate — Amount (clipped $500)", fontweight="bold")
sns.histplot(df.loc[df.Class==1, "Amount"].clip(upper=500), bins=50, color=PALETTE["fraud"], ax=axes[1], stat="density")
axes[1].set_title("Fraud — Amount (clipped $500)", fontweight="bold")
plt.suptitle("Real Amount Distribution by Class", fontweight="bold", color=PALETTE["accent"])
plt.tight_layout()
save_fig(fig, "02_amount_distribution.png")
plt.show()

Saved figure: nb1_results/figures/02_amount_distribution.png


In [5]:
# Transaction volume + fraud rate by hour-of-day (48h window)
df["hour"] = (df["Time"] // 3600) % 24
hourly = df.groupby("hour").agg(volume=("Class", "size"), fraud_rate=("Class", "mean")).reset_index()

fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.bar(hourly["hour"], hourly["volume"], color=PALETTE["legit"], alpha=0.6, label="Volume")
ax1.set_xlabel("Hour of day"); ax1.set_ylabel("Transaction volume", color=PALETTE["legit"])
ax2 = ax1.twinx()
ax2.plot(hourly["hour"], hourly["fraud_rate"]*100, color=PALETTE["fraud"], marker="o", linewidth=2, label="Fraud rate %")
ax2.set_ylabel("Fraud rate (%)", color=PALETTE["fraud"])
ax1.set_title("Real Transaction Volume & Fraud Rate by Hour of Day", fontweight="bold", color=PALETTE["accent"])
plt.tight_layout()
save_fig(fig, "03_hourly_volume_fraud.png")
plt.show()

Saved figure: nb1_results/figures/03_hourly_volume_fraud.png


In [6]:
# Correlation heatmap (V1-V28 + Amount) vs Class
corr_cols = [f"V{i}" for i in range(1, 29)] + ["Amount", "Class"]
corr = df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, cmap="RdBu_r", center=0, ax=ax, cbar_kws={"label": "Pearson correlation"})
ax.set_title("Real Feature Correlation Matrix", fontweight="bold", color=PALETTE["accent"])
plt.tight_layout()
save_fig(fig, "04_correlation_heatmap.png")
plt.show()

print("Top 8 features by absolute correlation with Class:")
display(corr["Class"].abs().sort_values(ascending=False)[1:9].to_frame("abs_corr_with_Class"))

Saved figure: nb1_results/figures/04_correlation_heatmap.png
Top 8 features by absolute correlation with Class:


,abs_corr_with_Class
V17,0.326481
V14,0.302544
V12,0.260593
V10,0.216883
V16,0.196539
V3,0.192961
V7,0.187257
V11,0.154876


## 2. Duplicate Investigation (Section 18.1)

In [7]:
dupes = df[df.duplicated(keep=False)]
dup_report = {
    "n_duplicate_rows": int(df.duplicated().sum()),
    "duplicates_by_class": dupes["Class"].value_counts().to_dict(),
}
print(dup_report)
print("Decision: duplicates retained for this run (PCA-anonymized features make a true-duplicate-vs-repeat-charge "
      "distinction unverifiable; dropping risks silently removing real repeat fraud attempts) — documented, not assumed.")

{'n_duplicate_rows': 1081, 'duplicates_by_class': {0: 1822, 1: 32}}
Decision: duplicates retained for this run (PCA-anonymized features make a true-duplicate-vs-repeat-charge distinction unverifiable; dropping risks silently removing real repeat fraud attempts) — documented, not assumed.


## 3. Class-Imbalance Technique Comparison (Section 6)

In [8]:
import class_imbalance_utils as ciu

feature_cols = [c for c in df.columns if c not in ("Time", "Class", "hour")]
X = df[feature_cols]
y = df["Class"]
amounts = df["Amount"].values

def build_model_fn(class_weight=None):
    return RandomForestClassifier(n_estimators=100, max_depth=12, random_state=RANDOM_SEED, n_jobs=-1, class_weight=class_weight)

print("Running real 5-fold comparison of class_weighting / threshold_moving / resampling_smote...")
t0 = time.time()
imbalance_results = ciu.compare_imbalance_strategies(X, y, build_model_fn, n_splits=5)
imbalance_summary = {name: {"mean_pr_auc": r.mean_pr_auc, "std_pr_auc": r.std_pr_auc} for name, r in imbalance_results.items()}
winner = ciu.winning_strategy(imbalance_results)
print(json.dumps(imbalance_summary, indent=2))
print("Winning strategy (real):", winner, f"[{time.time()-t0:.1f}s]")

fig, ax = plt.subplots(figsize=(7, 4.5))
names = list(imbalance_summary.keys())
means = [imbalance_summary[n]["mean_pr_auc"] for n in names]
stds = [imbalance_summary[n]["std_pr_auc"] for n in names]
colors = [PALETTE["accent"] if n == winner else PALETTE["grey"] for n in names]
ax.bar(names, means, yerr=stds, color=colors, capsize=6)
ax.set_ylabel("Mean CV PR-AUC")
ax.set_title("Real Class-Imbalance Strategy Comparison (winner highlighted)", fontweight="bold", color=PALETTE["accent"])
plt.tight_layout()
save_fig(fig, "05_imbalance_strategy_comparison.png")
plt.show()

Running real 5-fold comparison of class_weighting / threshold_moving / resampling_smote...


{
  "class_weighting": {
    "mean_pr_auc": 0.8326799369838532,
    "std_pr_auc": 0.0277526488733523
  },
  "threshold_moving": {
    "mean_pr_auc": 0.8448213812006662,
    "std_pr_auc": 0.02179241104041662
  },
  "resampling_smote": {
    "mean_pr_auc": 0.8294355819619776,
    "std_pr_auc": 0.02683132062996801
  }
}
Winning strategy (real): threshold_moving [1122.1s]
Saved figure: nb1_results/figures/05_imbalance_strategy_comparison.png


## 4. Stage A — Screen 5 Real Candidate Models (Section 6)

In [9]:
candidates = {
    "RandomForest": RandomForestClassifier(n_estimators=100, max_depth=12, random_state=RANDOM_SEED, n_jobs=-1),
    "RandomForest_Balanced": RandomForestClassifier(n_estimators=100, max_depth=12, random_state=RANDOM_SEED, n_jobs=-1, class_weight="balanced"),
    "XGBoost": XGBClassifier(random_state=RANDOM_SEED, eval_metric="aucpr", n_jobs=-1, n_estimators=200, max_depth=6),
    "LightGBM": LGBMClassifier(random_state=RANDOM_SEED, n_jobs=-1, verbosity=-1, n_estimators=200),
    "CatBoost": CatBoostClassifier(random_state=RANDOM_SEED, verbose=False, iterations=200),
}

import model_benchmark as mb
t0 = time.time()
stage_a = mb.run_stage_a_screening(X, y, candidates)  # list[StageAResult], sorted desc
stage_a_table = pd.DataFrame([{"model": r.candidate_name, "val_pr_auc": r.val_pr_auc} for r in stage_a])
print(f"[{time.time()-t0:.1f}s]")
display(stage_a_table.style.background_gradient(subset=["val_pr_auc"], cmap="Greens").set_caption("Stage A — Real Screening Results"))

fig, ax = plt.subplots(figsize=(7.5, 4.5))
colors = [PALETTE["accent"] if i < 2 else PALETTE["grey"] for i in range(len(stage_a_table))]
ax.barh(stage_a_table["model"], stage_a_table["val_pr_auc"], color=colors)
ax.invert_yaxis()
ax.set_xlabel("Validation PR-AUC (real, single split)")
ax.set_title("Stage A Screening — 5 Real Candidates (top 2 advance)", fontweight="bold", color=PALETTE["accent"])
plt.tight_layout()
save_fig(fig, "06_stage_a_screening.png")
plt.show()

[116.6s]


,model,val_pr_auc
0,CatBoost,0.871289
1,RandomForest,0.870140
2,RandomForest_Balanced,0.827268
3,XGBoost,0.789443
4,LightGBM,0.358361


Saved figure: nb1_results/figures/06_stage_a_screening.png


## 5. Stage B — Real 5-Fold Stratified CV: Champion + Runner-Up (Section 6)

In [10]:
top2_names = [r.candidate_name for r in stage_a[:2]]
top2 = {name: candidates[name] for name in top2_names}
print("Champion + runner-up advancing to real 5-fold stratified CV:", top2_names)

t0 = time.time()
stage_b = mb.run_stage_b_cv(X, y, top2, n_splits=5, n_bootstrap=1000)
champion_name = mb.select_champion(stage_b)
runner_up_name = [n for n in top2_names if n != champion_name][0]
print(f"CHAMPION (real, mean CV PR-AUC): {champion_name}")
print(f"RUNNER-UP: {runner_up_name}")
print(f"[{time.time()-t0:.1f}s]")

stage_b_table = pd.DataFrame([
    {"model": name, "fold": i+1, "pr_auc": v}
    for name, r in stage_b.items() for i, v in enumerate(r.fold_pr_auc)
])
display(stage_b_table.pivot(index="fold", columns="model", values="pr_auc")
        .style.background_gradient(cmap="Blues").set_caption("Real 5-Fold Stratified CV — Per-Fold PR-AUC"))

fig, ax = plt.subplots(figsize=(7.5, 5))
sns.boxplot(data=stage_b_table, x="model", y="pr_auc", ax=ax,
            palette={champion_name: PALETTE["accent"], runner_up_name: PALETTE["grey"]})
sns.stripplot(data=stage_b_table, x="model", y="pr_auc", ax=ax, color="black", size=7, jitter=0.05)
ax.set_title("Champion vs. Runner-Up — Real 5-Fold Stratified CV Distribution", fontweight="bold", color=PALETTE["accent"])
ax.set_ylabel("PR-AUC")
plt.tight_layout()
save_fig(fig, "07_champion_vs_runnerup_cv.png")
plt.show()

for name, r in stage_b.items():
    print(f"{name}: mean={r.mean_pr_auc:.4f}, 95% bootstrap CI={r.bootstrap_ci}")

Champion + runner-up advancing to real 5-fold stratified CV: ['CatBoost', 'RandomForest']


CHAMPION (real, mean CV PR-AUC): CatBoost
RUNNER-UP: RandomForest
[463.4s]


model,CatBoost,RandomForest
fold,,
1,0.835081,0.824966
2,0.882159,0.874711
3,0.849999,0.862529
4,0.832247,0.844890
5,0.841428,0.817012


Saved figure: nb1_results/figures/07_champion_vs_runnerup_cv.png
CatBoost: mean=0.8482, 95% bootstrap CI=(0.8161987442387733, 0.8783773017948813)
RandomForest: mean=0.8448, 95% bootstrap CI=(0.8149516458536813, 0.8738597283070101)


## 6. Temporal-Split Validation (Section 6, 19.4)

In [11]:
champion_model = top2[champion_name]
t0 = time.time()
temporal_pr_auc = mb.temporal_split_validation(df, "Time", feature_cols, "Class", champion_model)
print(f"Temporal-split PR-AUC: {temporal_pr_auc:.4f}  (real CV mean: {stage_b[champion_name].mean_pr_auc:.4f})  [{time.time()-t0:.1f}s]")

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.bar(["5-Fold CV\n(random)", "Temporal Split\n(time-ordered)"],
       [stage_b[champion_name].mean_pr_auc, temporal_pr_auc],
       color=[PALETTE["legit"], PALETTE["fraud"]])
ax.set_ylabel("PR-AUC")
ax.set_title(f"{champion_name} — CV vs. Temporal Validation (real, honest divergence)", fontweight="bold", color=PALETTE["accent"])
plt.tight_layout()
save_fig(fig, "08_cv_vs_temporal.png")
plt.show()

Temporal-split PR-AUC: 0.7692  (real CV mean: 0.8482)  [5.0s]
Saved figure: nb1_results/figures/08_cv_vs_temporal.png


## 7. Final Fit, Cost-Optimal Threshold & Confusion Matrix (Section 10)

In [12]:
t0 = time.time()
y_scores = cross_val_predict(champion_model, X, y, cv=5, method="predict_proba", n_jobs=-1)[:, 1]
champion_model.fit(X, y)
print(f"[{time.time()-t0:.1f}s]")

threshold_result = ciu.best_threshold_by_cost(y.values, y_scores, 4.41, 9.2, amounts)
CHOSEN_THRESHOLD = threshold_result["threshold"]
print(threshold_result)

# Cost-vs-threshold curve (real, vectorized recompute across a grid for visualization)
grid = np.linspace(0.001, 0.5, 200)
costs = []
for t in grid:
    preds = (y_scores >= t).astype(int)
    fn_cost = amounts[(y.values==1)&(preds==0)].sum() * 4.41
    fp_cost = amounts[(y.values==0)&(preds==1)].sum() * 9.2/100
    costs.append(fn_cost+fp_cost)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(grid, costs, color=PALETTE["accent"], linewidth=2)
ax.axvline(CHOSEN_THRESHOLD, color=PALETTE["fraud"], linestyle="--", label=f"Cost-optimal t={CHOSEN_THRESHOLD:.4f}")
ax.set_xlabel("Decision threshold"); ax.set_ylabel("Real total cost (EUR)")
ax.set_title("Real Cost-vs-Threshold Curve (sourced multipliers: $4.41 FN, 9.2x FP)", fontweight="bold", color=PALETTE["accent"])
ax.legend()
plt.tight_layout()
save_fig(fig, "09_cost_vs_threshold.png")
plt.show()

y_pred = (y_scores >= CHOSEN_THRESHOLD).astype(int)
cm = confusion_matrix(y, y_pred)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", ax=ax,
            xticklabels=["Pred: Legit", "Pred: Fraud"], yticklabels=["True: Legit", "True: Fraud"])
ax.set_title(f"Real Confusion Matrix @ t={CHOSEN_THRESHOLD:.4f}", fontweight="bold", color=PALETTE["accent"])
plt.tight_layout()
save_fig(fig, "10_confusion_matrix.png")
plt.show()

real_precision = precision_score(y, y_pred); real_recall = recall_score(y, y_pred)
print(f"Real precision: {real_precision:.4f}, real recall: {real_recall:.4f}")
benchmark_check = mb.compare_to_external_benchmark(real_precision, real_recall)
print(benchmark_check)

[34.7s]
{'threshold': 0.04430724903309097, 'total_cost': 65713.59124000002, 'fn_cost': 61408.720800000025, 'fp_cost': 4304.870439999996}


Saved figure: nb1_results/figures/09_cost_vs_threshold.png
Saved figure: nb1_results/figures/10_confusion_matrix.png


Real precision: 0.6964, real recall: 0.8252
{'your_precision': 0.6963979416809606, 'reference_precision': 0.9333, 'precision_gap': -0.23690205831903943, 'your_recall': 0.8252032520325203, 'reference_recall': 0.7467, 'recall_gap': 0.07850325203252029, 'investigate_flag': True, 'note': 'Large positive gaps may indicate leakage; large negative gaps may indicate a bug — investigate before trusting either.'}


## 8. Feature Importance — SHAP (Champion Model)

In [13]:
import shap
t0 = time.time()
sample_idx = np.random.RandomState(RANDOM_SEED).choice(len(X), size=5000, replace=False)
X_sample = X.iloc[sample_idx]

explainer = shap.TreeExplainer(champion_model)
shap_values = explainer.shap_values(X_sample)
if isinstance(shap_values, list):
    shap_values = shap_values[1]

fig = plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values, X_sample, show=False, plot_size=None)
plt.title(f"Real SHAP Summary — {champion_name} (5,000-row real sample)", fontweight="bold", color=PALETTE["accent"])
plt.tight_layout()
save_fig(plt.gcf(), "11_shap_summary.png")
plt.show()
print(f"[{time.time()-t0:.1f}s]  (SHAP computed on a 5,000-row real sample for tractability — disclosed, not hidden)")

Saved figure: nb1_results/figures/11_shap_summary.png
[1.1s]  (SHAP computed on a 5,000-row real sample for tractability — disclosed, not hidden)


## 9. Financial Impact — Real Measured Figures (Section 10)

In [14]:
total_fraud_amount = float(amounts[y.values==1].sum())
naive_no_model_cost = total_fraud_amount * 4.41
naive_05_preds = (y_scores >= 0.5).astype(int)
naive_05_cost = (amounts[(y.values==1)&(naive_05_preds==0)].sum()*4.41 +
                 amounts[(y.values==0)&(naive_05_preds==1)].sum()*9.2/100)
cost_optimal_cost = threshold_result["total_cost"]

impact_table = pd.DataFrame([
    {"Scenario": "No model (flag nothing)", "Total Cost (EUR)": naive_no_model_cost},
    {"Scenario": "Naive 0.5 threshold", "Total Cost (EUR)": naive_05_cost},
    {"Scenario": f"Cost-optimal threshold ({CHOSEN_THRESHOLD:.4f})", "Total Cost (EUR)": cost_optimal_cost},
])
display(impact_table.style.format({"Total Cost (EUR)": "€{:,.2f}"}).background_gradient(subset=["Total Cost (EUR)"], cmap="Reds_r"))

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.bar(impact_table["Scenario"], impact_table["Total Cost (EUR)"], color=[PALETTE["fraud"], PALETTE["grey"], PALETTE["legit"]])
ax.set_ylabel("Real total cost (EUR)")
ax.set_title("Real Financial Impact — Measured, Not Assumed", fontweight="bold", color=PALETTE["accent"])
plt.xticks(rotation=15, ha="right")
for i, v in enumerate(impact_table["Total Cost (EUR)"]):
    ax.annotate(f"€{v:,.0f}", (i, v), ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
save_fig(fig, "12_financial_impact.png")
plt.show()

print(f"Real savings vs. no model: EUR {naive_no_model_cost-cost_optimal_cost:,.2f}")
print(f"Real savings vs. naive 0.5 threshold: EUR {naive_05_cost-cost_optimal_cost:,.2f}")

,Scenario,Total Cost (EUR)
0,No model (flag nothing),"€265,164.35"
1,Naive 0.5 threshold,"€73,126.86"
2,Cost-optimal threshold (0.0443),"€65,713.59"


Saved figure: nb1_results/figures/12_financial_impact.png
Real savings vs. no model: EUR 199,450.76
Real savings vs. naive 0.5 threshold: EUR 7,413.27


## 10. Export All Results for Word / Excel / HTML Generation

In [15]:
final_results = {
    "run_metadata": {"random_seed": RANDOM_SEED, "n_rows": int(len(df)), "n_threads": _N_THREADS},
    "dataset": {"rows": int(len(df)), "columns": int(df.shape[1]), "duplicates": dup_report,
                "fraud_count": int(df["Class"].sum()), "fraud_rate": float(df["Class"].mean())},
    "imbalance_comparison": imbalance_summary,
    "imbalance_winner": winner,
    "stage_a": stage_a_table.to_dict(orient="records"),
    "stage_b": {name: {"mean_pr_auc": r.mean_pr_auc, "fold_pr_auc": r.fold_pr_auc, "bootstrap_ci": r.bootstrap_ci}
                for name, r in stage_b.items()},
    "champion_name": champion_name, "runner_up_name": runner_up_name,
    "temporal_pr_auc": temporal_pr_auc,
    "threshold_result": threshold_result,
    "real_precision": real_precision, "real_recall": real_recall,
    "benchmark_check": benchmark_check,
    "confusion_matrix": cm.tolist(),
    "financial_impact": impact_table.to_dict(orient="records"),
    "savings_vs_no_model": naive_no_model_cost-cost_optimal_cost,
    "savings_vs_naive_05": naive_05_cost-cost_optimal_cost,
}
with open(os.path.join(RESULTS_DIR, "nb1_final_results.json"), "w") as f:
    json.dump(final_results, f, indent=2, default=str)

with open(os.path.join(RESULTS_DIR, "champion_model.pkl"), "wb") as f:
    pickle.dump(champion_model, f)

print("Exported nb1_final_results.json and champion_model.pkl to", RESULTS_DIR)
print("Figures saved to", FIG_DIR, ":", sorted(os.listdir(FIG_DIR)))
print("\nNotebook 01 complete.")

Exported nb1_final_results.json and champion_model.pkl to nb1_results
Figures saved to nb1_results/figures : ['01_class_balance.png', '02_amount_distribution.png', '03_hourly_volume_fraud.png', '04_correlation_heatmap.png', '05_imbalance_strategy_comparison.png', '06_stage_a_screening.png', '07_champion_vs_runnerup_cv.png', '08_cv_vs_temporal.png', '09_cost_vs_threshold.png', '10_confusion_matrix.png', '11_shap_summary.png', '12_financial_impact.png']

Notebook 01 complete.
